# Talleres de Machine Learning — Aplicados a Detección de Fraude
## Dataset: Credit Card Fraud Detection (Kaggle)

Este notebook unifica los cuatro talleres del curso en un solo flujo coherente aplicado al dataset real de transacciones con tarjeta de crédito.

Cada bloque mantiene la estructura pedagógica establecida:
- **Antes del código**: explicación de qué se va a hacer.
- **Dentro del código**: comentarios en cada línea.
- **Después del código**: interpretación de lo observado.

---

## Estructura del notebook

| Parte | Tema | Bloques |
|---|---|---|
| **Preparación** | Librerías, carga y preprocesamiento | 1 – 3 |
| **Parte A** | SVM — Clasificador lineal | 4 – 6 |
| **Parte B** | SVM — Comparación de kernels | 7 – 9 |
| **Parte C** | K-Means — Clustering no supervisado | 10 – 14 |
| **Parte D** | Métricas supervisadas de clasificación | 15 – 20 |

---

## Contexto del problema

El dataset contiene **284,807 transacciones** con tarjeta de crédito de titulares europeos (septiembre 2013). Solo **492 (0.17%)** son fraudulentas. Las variables **V1–V28** son componentes PCA anonimizados; `Time`, `Amount` y `Class` (0 = legítima, 1 = fraude) son las variables originales.

---

---
# PREPARACIÓN
---

## Bloque 1. Importar librerías

En este bloque vamos a importar todas las librerías que usaremos a lo largo del notebook:
- manipulación de datos y arreglos numéricos,
- visualización,
- preprocesamiento,
- modelos de clasificación y clustering,
- y métricas de evaluación supervisada y no supervisada.

In [ ]:
# Importamos NumPy para arreglos y operaciones numéricas.
import numpy as np

# Importamos Pandas para cargar y explorar el dataset.
import pandas as pd

# Importamos Matplotlib para crear visualizaciones.
import matplotlib.pyplot as plt

# Importamos Seaborn para gráficas estadísticas más atractivas.
import seaborn as sns

# Importamos el escalador estándar para normalizar variables.
from sklearn.preprocessing import StandardScaler

# Importamos la función para dividir los datos en entrenamiento y prueba.
from sklearn.model_selection import train_test_split

# Importamos el clasificador SVM.
from sklearn.svm import SVC

# Importamos la regresión logística para la sección de métricas.
from sklearn.linear_model import LogisticRegression

# Importamos el algoritmo K-Means para clustering.
from sklearn.cluster import KMeans

# Importamos métricas de evaluación supervisada.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

# Importamos el índice silhouette para evaluar clustering.
from sklearn.metrics import silhouette_score

# Importamos resample para balancear clases.
from sklearn.utils import resample

# Fijamos la semilla global para reproducibilidad en todo el notebook.
np.random.seed(42)

# Confirmamos que el entorno está listo.
print('Librerías importadas correctamente.')

**Interpretación del bloque 1**

Este bloque único reemplaza los cuatro bloques de importación separados de los talleres individuales. Al centralizar todas las importaciones, el notebook queda organizado y evitamos repeticiones. Importar `KMeans` junto a `SVC` refleja que vamos a usar tanto aprendizaje **supervisado** (SVM, regresión logística) como **no supervisado** (K-Means) sobre el mismo dataset.

---

## Bloque 2. Cargar y explorar el dataset

En este bloque vamos a:
- cargar el archivo CSV desde el entorno de Colab,
- revisar su estructura, dimensiones y tipos de variables,
- y analizar el desbalance de clases, que es el principal reto de este dataset.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# OPCIÓN A — Subir el archivo manualmente a Colab.
# Descomenta las dos líneas siguientes y ejecútalas primero.
# from google.colab import files
# uploaded = files.upload()
# ──────────────────────────────────────────────────────────────────
# OPCIÓN B — Si el archivo está en tu Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# filepath = '/content/drive/MyDrive/creditcard.csv'
# ──────────────────────────────────────────────────────────────────
# OPCIÓN C — Ruta directa si ya está en el entorno.
filepath = 'creditcard.csv'

# Cargamos el dataset en un DataFrame de pandas.
df = pd.read_csv(filepath)

# Mostramos las dimensiones totales.
print('Shape del dataset:', df.shape)

# Mostramos los nombres de las columnas.
print('Columnas:', list(df.columns))

# Revisamos si hay valores nulos.
print('Valores nulos totales:', df.isnull().sum().sum())

# Mostramos la distribución de la variable objetivo.
conteo = df['Class'].value_counts()
print('\nDistribución de clases:')
print(conteo)
print(f'Porcentaje de fraude: {df["Class"].mean()*100:.3f}%')

# Mostramos las primeras 5 filas.
df.head()

In [ ]:
# Creamos una figura con dos subgráficas para explorar el dataset.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Gráfica 1: Desbalance de clases ────────────────────────────────
axes[0].bar(
    ['Legítima (0)', 'Fraude (1)'],
    [conteo[0], conteo[1]],
    color=['steelblue', 'tomato'], edgecolor='k'
)
axes[0].text(0, conteo[0]+3000, f'{conteo[0]:,}', ha='center', fontsize=11)
axes[0].text(1, conteo[1]+3000, f'{conteo[1]:,}', ha='center', fontsize=11)
axes[0].set_title('Distribución de clases — Desbalance extremo')
axes[0].set_ylabel('Número de transacciones')

# ── Gráfica 2: Distribución del monto por clase ────────────────────
axes[1].hist(df[df['Class']==0]['Amount'], bins=60, alpha=0.5,
             label='Legítima', color='steelblue', range=(0,2500))
axes[1].hist(df[df['Class']==1]['Amount'], bins=60, alpha=0.7,
             label='Fraude',   color='tomato',    range=(0,2500))
axes[1].set_title('Distribución del monto (Amount)')
axes[1].set_xlabel('Monto (€)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

**Interpretación del bloque 2**

El dataset tiene 284,807 transacciones y 31 columnas. No hay valores nulos. El desbalance es extremo: solo 0.173% son fraudes. La segunda gráfica revela que los fraudes tienden a montos relativamente bajos, lo que tiene sentido: los defraudadores hacen cargos pequeños para pasar desapercibidos. Este patrón es una señal que los modelos supervisados deberán aprender a detectar.

---

## Bloque 3. Preprocesamiento: escalado, balanceo y partición

En este bloque vamos a preparar los datos para todos los modelos del notebook:
- **escalar** las variables `Time` y `Amount` (las únicas no transformadas por PCA),
- **balancear** las clases mediante undersampling,
- y **dividir** en entrenamiento y prueba con `stratify`.

Este bloque se ejecuta **una sola vez** y el resultado se reutiliza en todas las partes siguientes.

In [ ]:
# Copiamos el dataset para no modificar el original.
df_proc = df.copy()

# Creamos el escalador estándar.
scaler = StandardScaler()

# Escalamos 'Amount': SVM calcula distancias, por lo que la escala importa.
df_proc['Amount'] = scaler.fit_transform(df_proc[['Amount']])

# Escalamos 'Time' a la misma escala que el resto de variables.
df_proc['Time'] = scaler.fit_transform(df_proc[['Time']])

# ── Undersampling ──────────────────────────────────────────────────
# Separamos las transacciones legítimas y las fraudulentas.
df_leg    = df_proc[df_proc['Class'] == 0]
df_fraud  = df_proc[df_proc['Class'] == 1]

# Reducimos las legítimas al mismo número que los fraudes.
df_leg_sample = resample(
    df_leg,
    replace=False,
    n_samples=len(df_fraud),
    random_state=42
)

# Combinamos y mezclamos el dataset balanceado.
df_bal = pd.concat([df_leg_sample, df_fraud]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print('Clases en el dataset balanceado:')
print(df_bal['Class'].value_counts())
print('Total de muestras:', len(df_bal))

# ── Separar X e y ─────────────────────────────────────────────────
# Separamos las variables de entrada y la variable objetivo.
X = df_bal.drop('Class', axis=1).values
y = df_bal['Class'].values

print('\nShape de X:', X.shape)
print('Shape de y:', y.shape)

# ── División entrenamiento / prueba ───────────────────────────────
# Dividimos en 70% entrenamiento y 30% prueba.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y          # Mantiene la proporción de clases en ambos subconjuntos.
)

print('\nShape de X_train:', X_train.shape)
print('Shape de X_test: ', X_test.shape)

**Interpretación del bloque 3**

Se aplicaron tres pasos de preprocesamiento esenciales:

1. **Escalado**: `Amount` varía entre 0 y 25,000€ mientras V1–V28 van de -3 a 3. Sin escalar, SVM daría un peso desproporcionado a `Amount`. El `StandardScaler` pone todo con media=0 y desviación=1.
2. **Undersampling**: De 284,315 transacciones legítimas, solo se usan 492 (igual al número de fraudes), creando un dataset de 984 filas perfectamente balanceado.
3. **Partición con stratify**: Garantiza que tanto el conjunto de entrenamiento como el de prueba tengan exactamente la misma proporción de fraudes (50/50 en este caso).

Este preprocesamiento es compartido por **todos los modelos del notebook**.

---

---
# PARTE A — SVM: Clasificador Lineal
---

## Bloque 4. Crear y entrenar el modelo SVM lineal

En este bloque vamos a:
- construir un clasificador SVM con kernel lineal,
- entrenarlo con el dataset balanceado,
- y dejarlo listo para hacer predicciones sobre el conjunto de prueba.

In [ ]:
# Creamos el modelo SVM con kernel lineal.
svm_linear = SVC(
    kernel='linear',         # Construye un hiperplano recto en el espacio de 30 dimensiones.
    C=1.0,                   # Parámetro de regularización: controla el margen vs errores.
    probability=True,        # Activa el cálculo de probabilidades (necesario para ROC-AUC).
    class_weight='balanced', # Penaliza más los errores en la clase minoritaria (fraude).
    random_state=42
)

# Entrenamos el modelo con los datos de entrenamiento.
svm_linear.fit(X_train, y_train)

# Confirmamos que el entrenamiento terminó.
print('Modelo SVM lineal entrenado correctamente.')
print('Número de vectores de soporte por clase:', svm_linear.n_support_)

**Interpretación del bloque 4**

El parámetro `C=1.0` controla el trade-off entre un **margen amplio** (menor C) y **pocos errores de entrenamiento** (mayor C). Con `class_weight='balanced'`, los errores sobre fraudes valen más que los errores sobre transacciones legítimas. Los **vectores de soporte** son los puntos que quedaron más cercanos al hiperplano y son los únicos que definen la frontera de decisión.

---

## Bloque 5. Hacer predicciones y evaluar el SVM lineal

En este bloque vamos a:
- predecir las clases del conjunto de prueba,
- calcular accuracy y ROC-AUC,
- y visualizar la matriz de confusión.

In [ ]:
# Generamos predicciones de clase sobre el conjunto de prueba.
y_pred_lin = svm_linear.predict(X_test)

# Generamos probabilidades para la clase positiva (fraude).
y_prob_lin = svm_linear.predict_proba(X_test)[:, 1]

# Calculamos accuracy (porcentaje global de aciertos).
acc_lin = accuracy_score(y_test, y_pred_lin)

# Calculamos ROC-AUC (más informativo que accuracy para clases desbalanceadas).
auc_lin = roc_auc_score(y_test, y_prob_lin)

# Mostramos los resultados.
print(f'Accuracy SVM lineal: {acc_lin:.4f}')
print(f'ROC-AUC  SVM lineal: {auc_lin:.4f}')

# Generamos el reporte completo de clasificación.
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred_lin, target_names=['Legítima', 'Fraude']))

In [ ]:
# Calculamos la matriz de confusión del SVM lineal.
cm_lin = confusion_matrix(y_test, y_pred_lin)

# Creamos la figura para el heatmap.
_, ax = plt.subplots(figsize=(5, 4))

# Dibujamos la matriz de confusión como heatmap.
sns.heatmap(
    cm_lin, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Legítima', 'Fraude'],
    yticklabels=['Legítima', 'Fraude'], ax=ax
)

# Etiquetamos el heatmap.
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — SVM Lineal')
plt.tight_layout()
plt.show()

**Interpretación del bloque 5**

La matriz de confusión organiza los resultados en cuatro celdas:
- **Verdaderos negativos (arriba izquierda)**: legítimas correctamente identificadas.
- **Falsos positivos (arriba derecha)**: legítimas clasificadas como fraude — costo: alarma falsa.
- **Falsos negativos (abajo izquierda)**: fraudes no detectados — **error más crítico**.
- **Verdaderos positivos (abajo derecha)**: fraudes correctamente detectados.

El **Recall de la clase Fraude** indica qué porcentaje de los fraudes reales fue detectado. Esa es la métrica más importante en este dominio.

---

## Bloque 6. Predicción sobre una nueva transacción (SVM lineal)

En este bloque vamos a crear una transacción ficticia y pedirle al modelo que decida si es legítima o fraudulenta, junto con la probabilidad de cada clase.

In [ ]:
# Creamos una transacción ficticia con los 30 atributos del dataset (escalados).
# Los valores corresponden a una transacción normal típica.
nueva_transaccion = np.array([[
     0.0,  -1.36, -0.07,  2.54,  1.38, -0.34,  0.46,  0.24,
     0.10,  0.36,  0.09, -0.55, -0.62, -0.99, -0.31,  1.47,
    -0.47,  0.21,  0.03,  0.40,  0.25, -0.02,  0.28, -0.11,
     0.07,  0.13, -0.19,  0.13, -0.02,  0.24
]])

# Predecimos la clase de la transacción.
pred_clase = svm_linear.predict(nueva_transaccion)[0]

# Obtenemos las probabilidades por clase.
prob_clases = svm_linear.predict_proba(nueva_transaccion)[0]

# Mostramos los resultados.
etiqueta = 'FRAUDE' if pred_clase == 1 else 'Legítima'
print(f'Clase predicha     : {pred_clase} → {etiqueta}')
print(f'P(Legítima)        : {prob_clases[0]:.4f}')
print(f'P(Fraude)          : {prob_clases[1]:.4f}')

**Interpretación del bloque 6**

Este bloque conecta el modelo con una decisión práctica concreta. La probabilidad de fraude puede usarse para definir umbrales de alerta: si `P(fraude) > 0.5` se bloquea la transacción; si `P(fraude) > 0.3` se marca para revisión manual. El SVM lineal entrega una frontera recta en el espacio de 30 dimensiones, lo que puede ser suficiente si los datos de fraude son aproximadamente separables linealmente tras la transformación PCA.

---

---
# PARTE B — SVM: Comparación de Kernels
---

## Bloque 7. Entrenar tres modelos SVM con kernels diferentes

En este bloque vamos a entrenar tres variantes de SVM sobre el mismo dataset:
- **kernel lineal**: frontera recta,
- **kernel polinómico**: frontera curva controlada por el grado,
- **kernel RBF**: fronteras locales muy flexibles.

Al usar el mismo split de entrenamiento/prueba, los resultados son directamente comparables.

In [ ]:
# Creamos el modelo con kernel lineal (ya entrenado en Parte A, lo reutilizamos).
svm_lin = svm_linear

# Creamos el modelo con kernel polinómico.
svm_poly = SVC(
    kernel='poly',           # Frontera polinómica.
    degree=3,                # Grado del polinomio.
    C=1.0,
    gamma='scale',           # Gamma escalado automáticamente según la varianza de X.
    probability=True,
    class_weight='balanced',
    random_state=42
)

# Creamos el modelo con kernel RBF (Radial Basis Function).
svm_rbf = SVC(
    kernel='rbf',            # Cada vector de soporte crea una 'burbuja' de influencia local.
    C=1.0,
    gamma='scale',
    probability=True,
    class_weight='balanced',
    random_state=42
)

# Entrenamos el kernel polinómico.
svm_poly.fit(X_train, y_train)
print('Kernel polinómico entrenado.')

# Entrenamos el kernel RBF.
svm_rbf.fit(X_train, y_train)
print('Kernel RBF entrenado.')

**Interpretación del bloque 7**

El kernel lineal ya fue entrenado en la Parte A, por lo que se reutiliza directamente. Los tres modelos usan `C=1.0`, `class_weight='balanced'` y `probability=True`, así que la única diferencia es la función de transformación del espacio. Esto nos permite **aislar el efecto del kernel** en el rendimiento final.

---

## Bloque 8. Comparar los tres kernels con tabla resumen y curvas ROC

En este bloque vamos a:
- calcular Accuracy y ROC-AUC para los tres kernels,
- mostrar una tabla comparativa,
- y graficar las tres curvas ROC en un mismo eje.

In [ ]:
# Generamos predicciones con los tres kernels.
pred_lin  = svm_lin.predict(X_test)
pred_poly = svm_poly.predict(X_test)
pred_rbf  = svm_rbf.predict(X_test)

# Generamos probabilidades para calcular ROC-AUC.
prob_lin  = svm_lin.predict_proba(X_test)[:, 1]
prob_poly = svm_poly.predict_proba(X_test)[:, 1]
prob_rbf  = svm_rbf.predict_proba(X_test)[:, 1]

# Calculamos las métricas para los tres kernels.
metricas_kernels = {
    'Lineal'    : (accuracy_score(y_test, pred_lin),  roc_auc_score(y_test, prob_lin)),
    'Polinómico': (accuracy_score(y_test, pred_poly), roc_auc_score(y_test, prob_poly)),
    'RBF'       : (accuracy_score(y_test, pred_rbf),  roc_auc_score(y_test, prob_rbf)),
}

# Construimos una tabla resumen.
df_res = pd.DataFrame(
    [(k, f'{v[0]:.4f}', f'{v[1]:.4f}') for k, v in metricas_kernels.items()],
    columns=['Kernel', 'Accuracy', 'ROC-AUC']
)

print('Comparación de kernels:')
print(df_res.to_string(index=False))

In [ ]:
# Calculamos los puntos de la curva ROC para cada kernel.
fpr_lin,  tpr_lin,  _ = roc_curve(y_test, prob_lin)
fpr_poly, tpr_poly, _ = roc_curve(y_test, prob_poly)
fpr_rbf,  tpr_rbf,  _ = roc_curve(y_test, prob_rbf)

# Extraemos los valores de AUC de las métricas ya calculadas.
auc_lin_v  = metricas_kernels['Lineal'][1]
auc_poly_v = metricas_kernels['Polinómico'][1]
auc_rbf_v  = metricas_kernels['RBF'][1]

# Creamos la figura para las curvas ROC.
_, ax = plt.subplots(figsize=(7, 5))

# Dibujamos la curva ROC del kernel lineal.
ax.plot(fpr_lin,  tpr_lin,  label=f'Lineal     (AUC={auc_lin_v:.4f})',  color='steelblue',   lw=2)

# Dibujamos la curva ROC del kernel polinómico.
ax.plot(fpr_poly, tpr_poly, label=f'Polinómico (AUC={auc_poly_v:.4f})', color='darkorange',  lw=2)

# Dibujamos la curva ROC del kernel RBF.
ax.plot(fpr_rbf,  tpr_rbf,  label=f'RBF        (AUC={auc_rbf_v:.4f})',  color='forestgreen', lw=2)

# Dibujamos la línea diagonal de referencia (clasificador aleatorio).
ax.plot([0,1],[0,1], 'k--', label='Aleatorio (AUC=0.5000)', lw=1)

# Configuramos la gráfica.
ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.set_title('Curvas ROC — Comparación de kernels SVM')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación del bloque 8**

La curva ROC muestra el trade-off entre detectar más fraudes (TPR alto) y generar más falsas alarmas (FPR alto). El área bajo la curva (AUC) resume este trade-off: cuanto más cercano a 1.0, mejor. La curva que se aleja más de la diagonal punteada corresponde al kernel más adecuado para este dataset. En datos transformados con PCA el kernel **RBF** suele liderar porque captura relaciones no lineales en el espacio de 30 dimensiones.

---

## Bloque 9. Análisis detallado del mejor kernel

En este bloque vamos a:
- identificar automáticamente el kernel con mayor ROC-AUC,
- revisar su matriz de confusión,
- y mostrar su reporte de clasificación completo.

In [ ]:
# Asociamos cada kernel con su modelo, predicciones y métricas.
kernels_info = [
    ('Lineal',      svm_lin,  pred_lin,  auc_lin_v),
    ('Polinómico',  svm_poly, pred_poly, auc_poly_v),
    ('RBF',         svm_rbf,  pred_rbf,  auc_rbf_v),
]

# Seleccionamos el kernel con mayor AUC.
best_name, best_model, best_pred, best_auc = max(kernels_info, key=lambda x: x[3])

print(f'Mejor kernel según ROC-AUC: {best_name} (AUC = {best_auc:.4f})')

# Calculamos la matriz de confusión del mejor kernel.
cm_best = confusion_matrix(y_test, best_pred)

print('\nMatriz de confusión:')
print(cm_best)

print('\nReporte de clasificación:')
print(classification_report(y_test, best_pred, target_names=['Legítima', 'Fraude']))

In [ ]:
# Visualizamos la matriz de confusión del mejor kernel.
_, ax = plt.subplots(figsize=(5, 4))

# Dibujamos el heatmap.
sns.heatmap(
    cm_best, annot=True, fmt='d', cmap='Greens',
    xticklabels=['Legítima', 'Fraude'],
    yticklabels=['Legítima', 'Fraude'], ax=ax
)

# Etiquetamos.
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusión — Mejor kernel: {best_name}')
plt.tight_layout()
plt.show()

**Interpretación del bloque 9**

Este bloque identifica el mejor modelo automáticamente. El análisis detallado de la matriz de confusión del ganador permite evaluar la tasa de fraudes no detectados (falsos negativos), que es el error más costoso en este dominio. Un **Recall de fraude alto** significa que el modelo detecta la mayoría de los intentos de robo, aunque pudiera generar algunas falsas alarmas adicionales.

---

---
# PARTE C — K-Means: Clustering No Supervisado
---

En esta parte aplicamos K-Means al dataset de fraude **sin usar la variable Class**. El objetivo es explorar si el algoritmo puede descubrir, por sí solo y sin supervisión, una estructura que separe aproximadamente las transacciones normales de las fraudulentas.

## Bloque 10. Preparar los datos para clustering

En este bloque vamos a:
- seleccionar las variables de entrada (sin `Class`),
- y reducir el dataset a dos dimensiones con PCA para poder visualizar los clusters.

In [ ]:
# Importamos PCA para reducir dimensiones y poder visualizar en 2D.
from sklearn.decomposition import PCA

# Usamos el dataset balanceado completo (sin dividir) para clustering.
# X ya fue definido en el Bloque 3 (sin la columna 'Class').
# y contiene las etiquetas reales, pero K-Means NO las usará.

# Reducimos a 2 dimensiones para visualización.
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

# Mostramos cuánta varianza explican las dos primeras componentes.
varianza = pca.explained_variance_ratio_
print(f'Varianza explicada por PC1: {varianza[0]*100:.2f}%')
print(f'Varianza explicada por PC2: {varianza[1]*100:.2f}%')
print(f'Total explicado           : {sum(varianza)*100:.2f}%')

# Visualizamos los datos reducidos coloreados según la clase REAL (solo como referencia).
_, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_2d[:,0], X_2d[:,1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.6, s=15)
ax.set_title('Datos en 2D (PCA) — coloreados por clase real (solo referencia)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

**Interpretación del bloque 10**

La reducción a 2D con PCA permite visualizar los datos antes del clustering. Las dos primeras componentes no capturan el 100% de la varianza (el dataset tiene 30 dimensiones), pero dan una idea de la distribución general. Si las dos clases aparecen relativamente separadas en 2D, es una señal positiva de que K-Means podría encontrar grupos con sentido.

---

## Bloque 11. Aplicar el método del codo para elegir k

En este bloque vamos a probar varios valores de k y observar cómo cambia la inercia para identificar el punto donde aumentar clusters ya no aporta mucha mejora.

In [ ]:
# Creamos una lista para guardar la inercia de cada valor de k.
inercias = []

# Definimos el rango de valores de k a probar.
rango_k = range(1, 11)

# Recorremos cada valor de k.
for k in rango_k:
    # Creamos un modelo K-Means con ese número de clusters.
    km_tmp = KMeans(
        n_clusters=k,
        init='k-means++',    # Inicialización inteligente: reduce la probabilidad de malos resultados.
        n_init=10,           # Número de veces que se repite con distintos centroides iniciales.
        random_state=42
    )
    # Entrenamos sobre los datos 2D para que la visualización sea consistente.
    km_tmp.fit(X_2d)
    # Guardamos la inercia: suma de distancias al cuadrado de cada punto a su centroide.
    inercias.append(km_tmp.inertia_)

# Graficamos la curva del codo.
_, ax = plt.subplots(figsize=(7, 5))
ax.plot(rango_k, inercias, marker='o', color='steelblue')
ax.set_title('Método del codo — Elegir el número óptimo de clusters (k)')
ax.set_xlabel('Número de clusters (k)')
ax.set_ylabel('Inercia')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación del bloque 11**

La curva del codo muestra cómo decrece la inercia al aumentar k. Se busca el punto donde la curva deja de caer bruscamente y se vuelve más plana: ese es el `k` óptimo. En este dataset esperamos ver un quiebre cerca de **k=2**, lo que coincide con las dos clases reales del problema (legítima y fraude). Si el codo aparece más tarde, significa que la estructura es más compleja en el espacio transformado.

---

## Bloque 12. Entrenar K-Means con k=2 y visualizar los clusters

En este bloque vamos a:
- entrenar K-Means con k=2 (las dos clases que esperamos encontrar),
- visualizar los clusters con sus centroides,
- y comparar visualmente con la distribución real.

In [ ]:
# Definimos k=2 porque esperamos que K-Means encuentre dos grupos naturales.
k_opt = 2

# Creamos el modelo K-Means con k=2.
kmeans = KMeans(
    n_clusters=k_opt,
    init='k-means++',    # Inicialización inteligente de centroides.
    n_init=10,           # 10 inicializaciones, se queda con la mejor.
    random_state=42
)

# Entrenamos K-Means sobre los datos 2D.
# IMPORTANTE: K-Means no recibe la variable 'y'; trabaja sin etiquetas.
kmeans.fit(X_2d)

# Guardamos las etiquetas de cluster asignadas a cada punto.
clusters = kmeans.labels_

# Guardamos las coordenadas de los centroides.
centroides = kmeans.cluster_centers_

print('Primeras 10 etiquetas asignadas por K-Means:')
print(clusters[:10])
print('\nCentroides encontrados:')
print(centroides)

In [ ]:
# Creamos una figura con dos subgráficas para comparar clusters vs realidad.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Gráfica 1: Clusters encontrados por K-Means ────────────────────
axes[0].scatter(
    X_2d[:,0], X_2d[:,1], c=clusters, cmap='coolwarm',
    edgecolors='k', alpha=0.6, s=15
)
# Dibujamos los centroides como puntos rojos grandes.
axes[0].scatter(
    centroides[:,0], centroides[:,1],
    c='black', s=250, marker='X', label='Centroides'
)
axes[0].set_title('Clusters encontrados por K-Means (sin etiquetas)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

# ── Gráfica 2: Clases reales (referencia) ─────────────────────────
axes[1].scatter(
    X_2d[:,0], X_2d[:,1], c=y, cmap='coolwarm',
    edgecolors='k', alpha=0.6, s=15
)
axes[1].set_title('Clases reales (referencia — no usadas para entrenar)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

**Interpretación del bloque 12**

Comparar las dos gráficas es el ejercicio pedagógico central de esta parte. Si los clusters de K-Means (izquierda) se parecen a las clases reales (derecha), significa que el algoritmo **descubrió la estructura del fraude sin haberla visto nunca**. Si hay diferencias, revela que la frontera entre fraude y transacción legítima no es perfectamente geométrica en este espacio. Esta comparación ilustra la diferencia fundamental entre aprendizaje **supervisado** (sabe las clases) y **no supervisado** (descubre estructura).

---

## Bloque 13. Evaluar la calidad del clustering

En este bloque vamos a usar dos métricas:
- **Inercia**: qué tan compactos son los clusters internamente.
- **Silhouette Score**: qué tan bien separados están los clusters entre sí.

In [ ]:
# Calculamos la inercia del modelo (ya disponible como atributo).
inercia = kmeans.inertia_

# Calculamos el índice silhouette.
silhouette = silhouette_score(X_2d, clusters)

# Mostramos las métricas.
print(f'Inercia del modelo    : {inercia:.2f}')
print(f'Silhouette Score      : {silhouette:.4f}')

print('\nInterpretación del Silhouette Score:')
if silhouette > 0.5:
    print('  → Clusters bien definidos y separados.')
elif silhouette > 0.25:
    print('  → Clusters con estructura moderada, algo de superposición.')
else:
    print('  → Clusters con superposición importante — la separación no es clara.')

**Interpretación del bloque 13**

- **Inercia**: mide la cohesión interna. Valores más bajos indican clusters más compactos. Sin un valor de referencia, se usa comparativamente entre distintos k.
- **Silhouette Score** (entre -1 y 1): mide tanto la cohesión interna como la separación entre clusters. Si el puntaje es bajo, significa que las transacciones fraudulentas y legítimas no forman grupos perfectamente separados en el espacio 2D, lo cual es esperado: el fraude fue diseñado para parecerse a transacciones normales.

---

## Bloque 14. Asignar nuevas transacciones a un cluster

En este bloque vamos a crear puntos nuevos en el espacio 2D y pedirle a K-Means que los asigne al cluster más cercano.

In [ ]:
# Creamos tres puntos ficticios en el espacio 2D de PCA.
nuevos_puntos_2d = np.array([
    [ 1.5,  0.5],   # Punto A — región de alta densidad
    [-2.0, -1.0],   # Punto B — región separada
    [ 0.0,  3.0],   # Punto C — zona de transición
])

# Predecimos el cluster de cada nuevo punto.
pred_nuevos = kmeans.predict(nuevos_puntos_2d)

# Mostramos los resultados.
print('Nuevos puntos y sus clusters asignados:')
for i, (punto, cluster) in enumerate(zip(nuevos_puntos_2d, pred_nuevos)):
    print(f'  Punto {chr(65+i)} {punto} → Cluster {cluster}')

# Visualizamos los nuevos puntos sobre los clusters existentes.
_, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_2d[:,0], X_2d[:,1], c=clusters, cmap='coolwarm',
           edgecolors='k', alpha=0.3, s=10)
ax.scatter(centroides[:,0], centroides[:,1],
           c='black', s=200, marker='X', label='Centroides')

# Dibujamos los nuevos puntos con etiqueta.
colores_nuevos = ['gold', 'limegreen', 'violet']
for i, (punto, cluster) in enumerate(zip(nuevos_puntos_2d, pred_nuevos)):
    ax.scatter(punto[0], punto[1], s=150, marker='*',
               color=colores_nuevos[i], edgecolors='k', zorder=5,
               label=f'Nuevo {chr(65+i)} → Cluster {cluster}')

ax.set_title('K-Means: asignación de nuevas observaciones')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Interpretación del bloque 14**

Este bloque muestra que K-Means, una vez entrenado, puede **clasificar nuevas observaciones** asignándolas al centroide más cercano. Esto tiene aplicaciones prácticas: si se detecta una transacción nueva cuyo perfil PCA cae en el cluster de comportamiento anómalo, puede marcarse para revisión. Esta capacidad convierte al clustering en una herramienta útil incluso sin etiquetas de clase.

---

---
# PARTE D — Métricas Supervisadas de Clasificación
---

En esta parte estudiamos en profundidad todas las métricas de evaluación supervisada usando un modelo de **Regresión Logística** entrenado sobre el mismo dataset balanceado. El objetivo es entender qué mide cada métrica y cuándo usarla.

## Bloque 15. Entrenar modelo base: Regresión Logística

En este bloque vamos a:
- entrenar una regresión logística sobre el dataset balanceado,
- obtener predicciones de clase y probabilidades,
- y dejar el modelo listo para calcular todas las métricas.

In [ ]:
# Creamos el modelo de regresión logística.
# Usamos max_iter=1000 para garantizar convergencia con 30 variables.
rl_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # Mismo criterio que en SVM para mantener consistencia.
    random_state=42
)

# Entrenamos sobre el mismo X_train / y_train del Bloque 3.
rl_model.fit(X_train, y_train)

# Generamos predicciones de clase (0 o 1).
y_pred_rl = rl_model.predict(X_test)

# Generamos probabilidades para la clase positiva (fraude = 1).
y_prob_rl = rl_model.predict_proba(X_test)[:, 1]

print('Modelo de Regresión Logística entrenado correctamente.')

**Interpretación del bloque 15**

La regresión logística se usa aquí como **modelo base** para estudiar métricas, no como el modelo principal del proyecto. Es un clasificador lineal que estima la probabilidad de fraude para cada transacción. `y_pred_rl` contiene las clases predichas (0 o 1), mientras que `y_prob_rl` contiene las probabilidades continuas necesarias para construir la curva ROC.

---

## Bloque 16. Calcular accuracy, precision, recall y F1-score

En este bloque vamos a calcular las cuatro métricas fundamentales de clasificación supervisada y entender qué pregunta responde cada una.

In [ ]:
# Calculamos accuracy: porcentaje total de predicciones correctas.
acc_rl  = accuracy_score(y_test, y_pred_rl)

# Calculamos precision: de las transacciones marcadas como fraude, ¿cuántas lo eran?
prec_rl = precision_score(y_test, y_pred_rl)

# Calculamos recall: de todos los fraudes reales, ¿cuántos detectamos?
rec_rl  = recall_score(y_test, y_pred_rl)

# Calculamos F1-score: media armónica entre precision y recall.
f1_rl   = f1_score(y_test, y_pred_rl)

# Mostramos los resultados.
print('─── Métricas — Regresión Logística ───')
print(f'Accuracy : {acc_rl:.4f}  → % total de aciertos')
print(f'Precision: {prec_rl:.4f}  → calidad de las alarmas de fraude')
print(f'Recall   : {rec_rl:.4f}  → cobertura de fraudes reales detectados')
print(f'F1-score : {f1_rl:.4f}  → balance entre precision y recall')

**Interpretación del bloque 16**

Cada métrica responde una pregunta diferente:

| Métrica | Pregunta que responde | Cuándo priorizarla |
|---|---|---|
| **Accuracy** | ¿Qué % total se acertó? | Clases balanceadas — no aplica aquí |
| **Precision** | ¿Qué % de las alarmas son reales? | Cuando falso positivo es muy costoso |
| **Recall** | ¿Qué % de fraudes reales detectamos? | **Detección de fraude — prioridad máxima** |
| **F1-score** | ¿Cómo balancear precision y recall? | Cuando ambas importan por igual |

En detección de fraude, el **Recall** es la métrica más crítica: dejar pasar un fraude (bajo recall) tiene un costo mayor que generar una falsa alarma (baja precision).

---

## Bloque 17. Visualizar la matriz de confusión

En este bloque vamos a construir y visualizar la matriz de confusión para entender en detalle qué tipo de errores comete el modelo.

In [ ]:
# Calculamos la matriz de confusión.
cm_rl = confusion_matrix(y_test, y_pred_rl)

# Mostramos la matriz en formato de texto.
print('Matriz de confusión (texto):')
print(cm_rl)

# Extraemos los cuatro valores para interpretarlos individualmente.
vn, fp, fn, vp = cm_rl[0,0], cm_rl[0,1], cm_rl[1,0], cm_rl[1,1]
print(f'\nVerdaderos Negativos (VN): {vn} — legítimas correctamente identificadas')
print(f'Falsos Positivos      (FP): {fp} — legítimas marcadas como fraude')
print(f'Falsos Negativos      (FN): {fn} — fraudes que pasaron desapercibidos ← ERROR CRÍTICO')
print(f'Verdaderos Positivos  (VP): {vp} — fraudes correctamente detectados')

# Creamos la visualización heatmap.
_, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_rl, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Legítima', 'Fraude'],
    yticklabels=['Legítima', 'Fraude'], ax=ax
)
ax.set_xlabel('Predicción')
ax.set_ylabel('Valor real')
ax.set_title('Matriz de confusión — Regresión Logística')
plt.tight_layout()
plt.show()

**Interpretación del bloque 17**

La matriz de confusión es la herramienta más completa para analizar errores. La diagonal principal (VN y VP) representa los aciertos; las celdas fuera de la diagonal representan errores. En fraude financiero, la celda de **Falsos Negativos** (fraudes no detectados) es la más crítica. Cuanto más pequeño sea ese número, más seguro es el sistema.

---

## Bloque 18. Construir la curva ROC y calcular el AUC

En este bloque vamos a:
- construir la curva ROC del modelo de regresión logística,
- calcular el AUC,
- y compararla visualmente con la curva del mejor kernel SVM.

In [ ]:
# Calculamos los puntos de la curva ROC para la regresión logística.
fpr_rl, tpr_rl, thresholds_rl = roc_curve(y_test, y_prob_rl)

# Calculamos el AUC.
auc_rl = roc_auc_score(y_test, y_prob_rl)
print(f'AUC — Regresión Logística  : {auc_rl:.4f}')
print(f'AUC — Mejor kernel SVM ({best_name}): {best_auc:.4f}')

# Creamos la figura comparativa.
_, ax = plt.subplots(figsize=(7, 5))

# Dibujamos la curva ROC de la regresión logística.
ax.plot(fpr_rl, tpr_rl,
        label=f'Regresión Logística (AUC={auc_rl:.4f})',
        color='steelblue', lw=2)

# Dibujamos la curva ROC del mejor kernel SVM para comparación.
best_prob = prob_lin if best_name=='Lineal' else prob_poly if best_name=='Polinómico' else prob_rbf
fpr_best, tpr_best, _ = roc_curve(y_test, best_prob)
ax.plot(fpr_best, tpr_best,
        label=f'SVM {best_name} (AUC={best_auc:.4f})',
        color='tomato', lw=2)

# Línea de referencia del clasificador aleatorio.
ax.plot([0,1],[0,1], 'k--', label='Aleatorio (AUC=0.50)', lw=1)

# Configuramos.
ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.set_title('Curva ROC — Regresión Logística vs Mejor SVM')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación del bloque 18**

La curva ROC muestra la capacidad discriminativa del modelo para **todos los umbrales posibles** de decisión, no solo el 0.5 por defecto. El eje X es la tasa de falsas alarmas (FPR) y el Y es la tasa de detección real (TPR/Recall). El modelo ideal tendría AUC=1.0 (curva que sube verticalmente). Comparar la regresión logística con el mejor kernel SVM permite decidir cuál modelo es más adecuado para este sistema de detección.

---

## Bloque 19. Resumen comparativo de todas las métricas y modelos

En este bloque vamos a construir una tabla final que consolida las métricas de todos los modelos entrenados en el notebook.

In [ ]:
# Calculamos las métricas completas para cada modelo supervisado.
modelos_eval = [
    ('SVM Lineal',      pred_lin,  prob_lin),
    ('SVM Polinómico',  pred_poly, prob_poly),
    ('SVM RBF',         pred_rbf,  prob_rbf),
    ('Reg. Logística',  y_pred_rl, y_prob_rl),
]

# Construimos la tabla de resultados.
filas = []
for nombre, pred, prob in modelos_eval:
    filas.append({
        'Modelo'    : nombre,
        'Accuracy'  : round(accuracy_score(y_test, pred), 4),
        'Precision' : round(precision_score(y_test, pred), 4),
        'Recall'    : round(recall_score(y_test, pred), 4),
        'F1-score'  : round(f1_score(y_test, pred), 4),
        'ROC-AUC'   : round(roc_auc_score(y_test, prob), 4),
    })

df_final = pd.DataFrame(filas)

# Mostramos la tabla.
print('═══ Resumen comparativo de todos los modelos ═══')
print(df_final.to_string(index=False))

# Identificamos el mejor modelo según ROC-AUC.
mejor_idx = df_final['ROC-AUC'].idxmax()
print(f"\nMejor modelo según ROC-AUC: {df_final.loc[mejor_idx, 'Modelo']} ({df_final.loc[mejor_idx, 'ROC-AUC']})")

**Interpretación del bloque 19**

Esta tabla permite comparar los cuatro modelos bajo las mismas condiciones (mismo dataset, mismo split). Las conclusiones clave:
- **Accuracy** puede parecer similar entre modelos, pero oculta diferencias importantes en Recall.
- **Recall** es la métrica más crítica: un modelo con alto Recall detecta más fraudes reales.
- **ROC-AUC** es la métrica más robusta para comparar modelos sin depender de un umbral fijo.
- El modelo con mayor ROC-AUC es el más recomendable para producción.

---

## Bloque 20. Predicción sobre una nueva transacción con análisis completo

En este bloque vamos a aplicar el mejor modelo del notebook (por ROC-AUC) para evaluar una transacción nueva, mostrando tanto la clase predicha como las probabilidades de cada clase.

In [ ]:
# Tomamos la primera observación del conjunto de prueba como caso real de ejemplo.
caso_real = X_test[0].reshape(1, -1)
etiqueta_real = y_test[0]

print('=== Análisis con todos los modelos supervisados ===')
print(f'Clase REAL de esta transacción: {etiqueta_real} ({"Fraude" if etiqueta_real==1 else "Legítima"})')
print()

# Evaluamos la transacción con cada modelo supervisado.
for nombre, modelo in [
    ('SVM Lineal',     svm_lin),
    ('SVM Polinómico', svm_poly),
    ('SVM RBF',        svm_rbf),
    ('Reg. Logística', rl_model),
]:
    # Predecimos la clase.
    pred  = modelo.predict(caso_real)[0]
    # Obtenemos la probabilidad de fraude.
    proba = modelo.predict_proba(caso_real)[0][1]
    # Determinamos si el modelo acertó.
    acierto = '✓' if pred == etiqueta_real else '✗'
    print(f'{acierto} {nombre:18s} → Predicción: {pred} ({'Fraude' if pred==1 else 'Legítima':9s}) | P(fraude)={proba:.4f}')

**Interpretación del bloque 20**

Este bloque cierra el flujo completo del notebook aplicando todos los modelos supervisados a un caso real del conjunto de prueba. La columna `✓/✗` indica si cada modelo acertó. La probabilidad de fraude muestra el nivel de confianza de cada predicción: valores cercanos a 1.0 indican que el modelo está muy seguro de que es fraude; valores cercanos a 0.5 indican incertidumbre. En un sistema de producción, esta probabilidad puede usarse para decidir si bloquear automáticamente, enviar a revisión manual, o aprobar la transacción.

---

---
# CIERRE DEL NOTEBOOK
---

## Resumen de aprendizajes

| Tema | Concepto clave aprendido |
|---|---|
| **Preprocesamiento** | Escalar y balancear clases no es opcional en SVM — define si el modelo funciona |
| **SVM Lineal** | Construye el hiperplano de máximo margen en el espacio de características |
| **Kernels** | Lineal < Polinómico < RBF en complejidad y flexibilidad — el kernel correcto depende del problema |
| **K-Means** | Descubre estructura sin etiquetas — útil como exploración y para sistemas sin supervisión |
| **Métricas** | Accuracy engaña en clases desbalanceadas — ROC-AUC y Recall son las métricas correctas para fraude |

## Flujo completo aplicado
```
Dataset (284,807 transacciones)
    ↓
Preprocesamiento (escalado + undersampling + split)
    ↓
┌─────────────────────────────────────────────────┐
│  SUPERVISADO          │  NO SUPERVISADO          │
│  SVM (3 kernels)      │  K-Means (k=2)           │
│  Reg. Logística       │  Silhouette + Codo       │
├───────────────────────┴──────────────────────────┤
│  Métricas: Accuracy · Precision · Recall · F1    │
│            Confusion Matrix · ROC-AUC            │
└──────────────────────────────────────────────────┘
    ↓
Predicción sobre nuevas transacciones
```